# <span style="color:#1D4ED8">Tabular Q-learning | 45-minute experiential coaching session</span>
**ALU — SE Specialisation Coach (Machine Learning)**

**Only live notebook.** Prepare separately using the 15-minute pre-session notebook. In this class, you will **predict → discuss → calculate → debug in pairs → test → inspect learning → reflect**.

**Outcomes (observable):** (1) label agent/state/action/reward; (2) compute a Q update; (3) fix two conceptual bugs without editing the original cell and pass two checks; (4) interpret exploration and a learned policy.

**Time map:** 0–5 challenge/recap · 5–10 environment/Q-table · 10–17 update · 17–30 peer debugging · 30–37 learning demo · 37–42 policy critique · 42–45 exit ticket.

**Facilitation:** Students work in driver–navigator pairs; switch roles after the first correction. Coach asks questions and gathers evidence rather than lecturing.

## <span style="color:#1D4ED8">Open the live notebook before the timer starts</span>
Use https://colab.research.google.com/ → Upload → select this notebook → Connect; or JupyterLab/VS Code with Python 3 and `numpy`, `pandas`, `matplotlib` installed. Run code cells top-to-bottom with **Shift+Enter**. If connectivity fails, work from a printed grid/Q-table and explain corrections verbally. **Opening and connecting should happen before the 45-minute session.**

## <span style="color:#1D4ED8">0–5 min · 3-minute recap (0–3) + 2-minute robot challenge (3–5)</span>

**0–1 min (everyone):** Observe the grid below. Point out **agent**, **state**, **actions** and **goal**. The coach briefly defines any unfamiliar word. **No preparation is required to join.**

**1–2 min (pair):** Look at this **illustrative**, not learned, Q-table row for the Start state:

| State | Up | Down | Left | Right |
|---|---:|---:|---:|---:|
| Start (0,0) | 0.0 | 1.0 | 0.0 | **2.0** |

Predict which action a greedy policy would choose. **Answer: Right** (highest estimated value), but the estimate might be inaccurate.

**2–3 min (pair):** If `gamma = 0`, does the Q-learning target value estimated future rewards? **No; only the immediate reward.** Ask an unprepared learner to explain the reasoning in their own words.

**3–5 min (challenge):** The delivery robot starts at **(0,0)** on a 3×3 grid, must reach **(2,2)** and cannot enter the centre **(1,1)**. `S` = start, `#` = wall, `G` = goal. Moves are up/down/left/right; every non-goal step (including invalid moves) costs **−1**; reaching the goal earns **+10**. Invalid moves leave the robot in place. Predict a route, then run the next cell.

**Coach timing rule:** The recap is exactly **0–3 min within the 45-minute live session**, not an extra activity. The robot challenge occupies **3–5 min**. Ask for predictions before revealing outputs; do not extend the recap.

In [ ]:
SIZE = 3
START, WALL, GOAL = (0, 0), (1, 1), (2, 2)
ACTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]
ACTION_NAMES = ["Up", "Down", "Left", "Right"]
for r in range(SIZE):
    print(" ".join("S" if (r,c)==START else "#" if (r,c)==WALL else "G" if (r,c)==GOAL else "·" for c in range(SIZE)))

## <span style="color:#1D4ED8">5–10 min · Translate the story into reinforcement learning</span>

**Agent:** the robot making decisions. **Environment:** the grid and transition rules. **State:** its current row/column. **Action:** one of four moves. **Reward:** feedback from a move. **Episode:** a sequence ending at the goal or a time limit. **Policy:** a rule for choosing actions.

The **Q-table** stores an estimated discounted return for each `(state, action)` pair. Unlike a supervised classifier trained on labelled correct moves, this learner improves its action values from experienced transitions.

**Pair prompt:** What information is missing if the robot's true task also depends on whether it is carrying a parcel? A state must include information needed to predict rewards and transitions (the Markov property).

In [ ]:
def step(state, action):
    nr = state[0] + ACTIONS[action][0]
    nc = state[1] + ACTIONS[action][1]
    nxt = (nr, nc)
    if not (0 <= nr < SIZE and 0 <= nc < SIZE) or nxt == WALL:
        nxt = state
    done = (nxt == GOAL)
    reward = 10.0 if done else -1.0
    return nxt, reward, done

print("Right from start:", step(START, 3))
print("Up from start (invalid):", step(START, 0))
print("Into goal:", step((2,1), 3))

### <span style="color:#1D4ED8">The Q-table: the central learning artifact</span>

Each **row** represents a valid robot position (state), and each **column** represents an action. The value in a cell, **Q(state, action)**, estimates the discounted future reward of taking that action and then acting well. **It is an estimate, not a probability or an immediate reward.**

| State | Up | Down | Left | Right |
|:--|--:|--:|--:|--:|
| (0,0) Start | 0 | 0 | 0 | 0 |
| (0,1) | 0 | 0 | 0 | 0 |
| (0,2) | 0 | 0 | 0 | 0 |
| (1,0) | 0 | 0 | 0 | 0 |
| (1,2) | 0 | 0 | 0 | 0 |
| (2,0) | 0 | 0 | 0 | 0 |
| (2,1) | 0 | 0 | 0 | 0 |
| (2,2) Goal | 0 | 0 | 0 | 0 |

The centre (1,1) is blocked, so it is not a valid state. The goal is terminal: its row is a **placeholder**, not a set of actions the robot takes after arrival. Invalid moves are still available to the policy, but leave the robot in the same state with a step penalty.

**Predict:** At initialization, which action should the robot prefer? Since every value is tied, it has no learned preference yet.

**Continuity with preparation:** The pre-session table was an illustrative snapshot; the next cell shows the **actual initial table of zeros** used for this environment. Do not confuse example values with trained results.

In [ ]:
import pandas as pd
STATES = [(r,c) for r in range(SIZE) for c in range(SIZE) if (r,c) != WALL]
ACTION_NAMES = ["Up", "Down", "Left", "Right"]
initial_q = {state: [0.0] * len(ACTION_NAMES) for state in STATES}

def show_qtable(table, decimals=2):
    frame = pd.DataFrame.from_dict(table, orient="index", columns=ACTION_NAMES)
    frame.index = [f"{s}" + (" (start)" if s==START else " (terminal)" if s==GOAL else "") for s in frame.index]
    frame.index.name = "State (row, column)"
    return frame.round(decimals)

display(show_qtable(initial_q))

**Coach check:** Why is there no row for `(1,1)`? Why are all four actions present at `(0,0)` even though `Up` and `Left` would hit the boundary?

## <span style="color:#0F766E">10–17 min · From the prep Q-table to a visible update</span>

**Retrieve from prep:** What are the old estimate, learning rate, target and next-state value? Use the **same illustrative Q-table from preparation**: at Start `(0,0)`, `Right` has value **2.0**; moving Right leads to `(0,1)`, where the highest displayed action value is **6.0** (Right). This nonterminal movement earns **−1**. These illustrative estimates are supplied for arithmetic practice, **not** learned training results.

For a **nonterminal** transition, Q-learning updates its estimate with

\[
Q(s,a)\leftarrow Q(s,a)+\alpha\left[r+\gamma\max_{a'}Q(s',a')-Q(s,a)\right].
\]

Here, \(\alpha\) is the learning rate, \(\gamma\) discounts future reward, \(r\) is the immediate reward, and \(s'\) is the **next** state.

For a **terminal** transition, there is no future action to bootstrap from:

\[
Q(s,a)\leftarrow Q(s,a)+\alpha\left[r-Q(s,a)\right].
\]

**Individual prediction (2 minutes):** Suppose \(Q(s,a)=2\), \(\alpha=0.5\), \(r=-1\), \(\gamma=0.9\), and the best next-state value is \(6\). Calculate the new Q-value before running the cell. Then pair-check.

**Coach questions:** Why use \(Q(s',a')\), not \(Q(s,a')\), for the target? What changes when the next state is terminal?

In [ ]:
def q_update(old_q, reward, next_best, alpha, gamma, done):
    target = reward if done else reward + gamma * next_best
    return old_q + alpha * (target - old_q)

print("Nonterminal update:", q_update(2, -1, 6, .5, .9, False))
print("Terminal update:", q_update(2, 10, 999, .5, .9, True))
assert q_update(2, -1, 6, .5, .9, False) == 3.2
assert q_update(2, 10, 999, .5, .9, True) == 6.0

### <span style="color:#0F766E">Update one visible Q-table cell</span>

Use the same nonterminal numerical example from the previous calculation. This reuses the **exact illustrative Q-table from preparation**, not a trained Q-table. The next-state maximum **6.0** is read from the `(0,1)` row, Right column. Watch the `(0,0), Right` entry change from `2.0` to `3.2` while other entries remain unchanged.

**Pair task:** Predict the updated cell, execute the code, and explain why the next-state maximum is used. Then change `reward` to `-2` and predict the new value **from the original 2.0**.

In [ ]:
# The SAME illustrative Q-table used in the 15-minute preparation.
example_q = {state: [0.0]*4 for state in STATES}
example_q[(0,0)] = [0.0, 1.0, 0.0, 2.0]
example_q[(0,1)] = [1.0, -2.0, 0.5, 6.0]
example_q[(2,1)] = [0.0, 4.0, 1.0, 7.0]
print("BEFORE (illustrative Q-table; NOT trained output):")
display(show_qtable(example_q))
next_state, reward, done = step(START, 3)  # Right from (0,0)
assert next_state == (0,1) and reward == -1 and not done
next_best = max(example_q[next_state])
assert next_best == 6.0
example_q[START][3] = q_update(old_q=example_q[START][3], reward=reward,
                               next_best=next_best, alpha=0.5, gamma=0.9, done=done)
print("AFTER (only Start × Right changes):")
display(show_qtable(example_q))
assert abs(example_q[START][3] - 3.2) < 1e-10

## <span style="color:#7C3AED">17–30 min | Peer pair programming: debug the update (13 minutes)</span>
**Challenge:** You are given a complete, runnable but **conceptually incorrect** Q-learning update. The original cell is evidence and must remain unchanged.

- **17–19 (2 min):** Independently predict correct nonterminal and terminal outputs; compare with your partner.
- **19–23 (4 min):** Run the **original** cell unchanged. Copy its complete code into the **new empty working code cell directly below**. Diagnose bug 1. Switch driver/navigator roles and diagnose bug 2.
- **23–27 (4 min):** Correct only the working copy and run the two tests. Explain failures before retrying.
- **27–30 (3 min):** Each partner explains one correction to another pair or the coach.

**NON-NEGOTIABLE: NEVER EDIT THE ORIGINAL PROBLEM CELL.** You may rerun it; all edits belong in the working copy.

### <span style="color:#7C3AED">17–19 min · Predict before executing</span>
For a **nonterminal** transition, the Q-learning update is

$$Q(s,a)\leftarrow Q(s,a)+\alpha\left[r+\gamma\max_{a'}Q(s',a')-Q(s,a)\right].$$

For a **terminal** transition, there is no future action to take, so the target is simply $r$.

**Your prediction:** Let $Q(s,a)=2$, $\alpha=0.5$, $\gamma=0.9$, $r=-1$, and $\max_{a'}Q(s',a')=6$. What should the new Q-value be? What if the transition is terminal and $r=10$?

**Write your predictions here before running code:**
- Nonterminal updated value: **_____**
- Terminal updated value: **_____**
- Why is the terminal case different? **_____**

**Trace the source of 6:** it is the highest illustrative Q-value in the **next state `(0,1)`**, not the maximum of the current state `(0,0)`.

### <span style="color:#7C3AED">19–23 min · Run, copy, then diagnose (pair programming)</span>

**Step 1 — RUN the next cell exactly as supplied.** It contains a complete, deliberately incorrect Q-learning update. Observe both outputs and discuss why they disagree with the mathematical targets.

**Step 2 — NEVER CORRECT OR EDIT THE ORIGINAL PROBLEM CELL.** Keep it unchanged as your reference.

**Step 3 — In the NEW, EMPTY CODE CELL directly underneath the original, copy all the original code.** Rename the copied function `student_q_update` (leave `broken_q_update` untouched). Then fix the two conceptual errors in the *copied* code only. The verification tests expect `student_q_update`.

**Driver:** Copy the code and investigate the first error. **Navigator:** Explain the next-state target. Switch roles for the terminal-state correction.

**Questions:** (a) Should the target use the best value of the current state or the next state? (b) When `done=True`, should the update include a future-state value?

In [ ]:
# ORIGINAL BROKEN PROBLEM — RUN THIS CELL, BUT NEVER EDIT OR CORRECT IT.
def broken_q_update(old_q, reward, current_state_best, next_state_best, alpha, gamma, done):
    # Estimate the learning target.
    target = reward + gamma * current_state_best

    # Handle the end of an episode.
    if done:
        target = reward + gamma * next_state_best

    return old_q + alpha * (target - old_q)

print('Original nonterminal:', broken_q_update(2, -1, 9, 6, 0.5, 0.9, False))
print('Original terminal:   ', broken_q_update(2, 10, 9, 6, 0.5, 0.9, True))

In [ ]:
# NEW WORKING CELL — STUDENTS: COPY the entire ORIGINAL code above INTO THIS CELL.
# Rename the copied function to student_q_update.
# Correct the two conceptual errors ONLY IN THIS CELL.
# Do not edit the original cell. Run this cell before running the tests below.


### <span style="color:#7C3AED">23–27 min · Test your corrected copy</span>

After you have copied and corrected the original code in the **new working cell immediately above**, run the tests below. They intentionally fail until your corrected function is defined and both bugs are fixed. **Do not modify the original problem cell or the verification tests.**

Expected results: nonterminal **3.2**, terminal **6.0**. Compare these against the original printed outputs. If the tests fail, return to your working copy and explain the target calculation to your partner.

In [ ]:
# These checks intentionally FAIL until the WORKING COPY is corrected. The original remains unchanged.
assert abs(student_q_update(2, -1, 9, 6, 0.5, 0.9, False) - 3.2) < 1e-10, 'Nonterminal target uses the wrong state.'
assert abs(student_q_update(2, 10, 9, 6, 0.5, 0.9, True) - 6.0) < 1e-10, 'Terminal transition must not bootstrap.'
print('Both checks passed. Explain each fix to your partner.')

### <span style="color:#7C3AED">27–30 min · Explain and transfer</span>
**Peer teach-back (60 seconds each):**
- Point to the exact line you changed and describe the original error.
- Connect the repaired code to the mathematical update.
- Explain what the Q-table stores and why incorrect targets could distort learned action values.

**Evidence to share with the coach:** Both tests pass, and each partner can justify one correction.

**Extension if you finish early:** What would happen if `alpha = 0`? Would the Q-table change? Why?

**If stuck:** Use the hint below before asking for the answer.

**Hint (only after partner discussion):**
For a nonterminal step, bootstrap from the maximum Q-value of the *next* state. For a terminal step, use the immediate reward as the target; there is no next action.


**Next:** The coach demonstrates a full training loop using the same Q-learning principle.

## <span style="color:#B45309">30–37 min | Watch learning emerge (7 minutes)</span>
**30–32 (2 min):** Pairs predict: how will the Q-table change after repeated episodes? Which actions are exploratory?

**32–35 (3 min):** Coach runs the training cell below and displays the **all-zero training initialization versus trained Q-table**. Students identify the best-valued action at the start and check whether the goal is reached.

**35–37 (2 min):** Pairs compare `epsilon=0` and `epsilon=0.25` (and optionally `gamma=0` versus `0.9`) using the compact comparison cell below. Report **one observed number and one limitation**. If time is short, the coach runs it while students interpret.

**Reminder:** A single run is evidence of behaviour, not proof of convergence or optimality.

In [ ]:
# FACILITATOR DEMONSTRATION — run once after pair debugging.
import random
import numpy as np
import matplotlib.pyplot as plt

VALID_STATES = [(r,c) for r in range(SIZE) for c in range(SIZE) if (r,c) != WALL]

def train(episodes=2000, alpha=0.3, gamma=0.9, epsilon=0.25, seed=7, max_steps=30):
    rng = random.Random(seed)
    q = {state: [0.0]*4 for state in VALID_STATES}
    rewards, success_flags = [], []
    for episode in range(episodes):
        state, total, success = START, 0.0, False
        for _ in range(max_steps):
            if rng.random() < epsilon:
                action = rng.randrange(4)
            else:
                best = max(q[state])
                action = rng.choice([a for a,v in enumerate(q[state]) if v == best])
            nxt, reward, done = step(state, action)
            q[state][action] = q_update(q[state][action], reward,
                                          max(q[nxt]), alpha, gamma, done)
            total += reward
            state = nxt
            if done:
                success = True
                break
        rewards.append(total)
        success_flags.append(success)
    return q, np.array(rewards), np.array(success_flags)

q, rewards, successes = train()
print("Episodes:", len(rewards))
print("Successes:", int(successes.sum()))
print("Mean reward, first 100:", round(float(rewards[:100].mean()), 2))
print("Mean reward, last 100:", round(float(rewards[-100:].mean()), 2))
print("Q-values at start:", dict(zip(ACTION_NAMES, np.round(q[START], 2))))
assert len(rewards) == 2000

### <span style="color:#B45309">Inspect the trained Q-table and derive the policy</span>

Now compare the **all-zero initialization of the actual training run** with the learned values. This is distinct from the *illustrative* pre-session Q-table used for hand calculations. For each **nonterminal** state, a greedy policy selects the action with the highest Q-value. The highest-valued action may still be suboptimal if learning was insufficient. The terminal state's row is excluded from policy selection.

**Partner discussion:** Find the start-state row. Which action has the highest value? Does this match your original route prediction? Explain why some entries can remain negative or less well estimated.

In [ ]:
print("INITIAL Q-TABLE:")
display(show_qtable(initial_q))
print("TRAINED Q-TABLE:")
display(show_qtable(q))
policy_rows = []
for state in STATES:
    if state == GOAL:
        policy_rows.append((str(state), "Terminal — no action", None))
    else:
        best_action = max(range(4), key=lambda a: q[state][a])
        policy_rows.append((str(state), ACTION_NAMES[best_action], round(q[state][best_action], 2)))
policy_df = pd.DataFrame(policy_rows, columns=["State", "Greedy action", "Highest Q-value"])
display(policy_df)
assert len(policy_df)==8

**Interpretation caution:** A single training run illustrates a learned policy, not a proof of optimality. The Q-table has only eight reachable states here; larger or continuous state spaces make tabular storage impractical.

### <span style="color:#B45309">Optional extension / coach-led comparison (within 35–37 min if time permits)</span>
Predict which parameter controls exploration (`epsilon`) and which weights future rewards (`gamma`). The coach runs the comparison; pairs interpret **one** result. **Do not delay the exit ticket** for this extension. Random tie-breaking means `epsilon=0` does not guarantee identical action choices.

In [ ]:
settings = [
    ("No epsilon exploration", 0.0, 0.9),
    ("Epsilon 0.25", 0.25, 0.9),
    ("Myopic gamma 0", 0.25, 0.0),
    ("Future-aware gamma 0.9", 0.25, 0.9),
]
for label, eps, gam in settings:
    _, rr, ss = train(epsilon=eps, gamma=gam, seed=7)
    print(f"{label:25s} | success={ss.mean():.1%} | last100 mean={rr[-100:].mean():6.2f}")

### <span style="color:#B45309">Optional evidence visualization (run after class if time is tight)</span>
This moving-average reward plot is supplementary. The live coaching priority is explaining the Q-table, comparing exploration and completing the exit ticket.

In [ ]:
window = 100
moving_mean = np.convolve(rewards, np.ones(window)/window, mode="valid")
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(np.arange(window, len(rewards)+1), moving_mean)
ax.set(xlabel="Episode", ylabel="Mean reward (100 episodes)",
       title="Training reward trend with ongoing exploration")
fig.tight_layout()
plt.show()

## <span style="color:#15803D">37–42 min | Explain and critique the learned policy (5 minutes)</span>
**37–39:** Pairs inspect the trained Q-table and trace the greedy route.

**39–41:** Each partner explains why the robot avoids the wall and when the episode ends. Identify a limitation: small discrete state space, sensitivity to training settings, or lack of convergence guarantee.

**41–42:** Coach asks one pair to share evidence; another pair challenges the interpretation. The route visualization below can be run by the coach.

In [ ]:
def greedy_route(q_table, max_steps=15):
    state, route = START, [START]
    for _ in range(max_steps):
        action = max(range(4), key=lambda a: q_table[state][a])
        state, reward, done = step(state, action)
        route.append(state)
        if done:
            break
    return route

route = greedy_route(q)
print("Greedy route:", route)
print("Goal reached:", route[-1] == GOAL)
print("Number of moves:", len(route)-1)

fig, ax = plt.subplots(figsize=(4, 4))
for r in range(SIZE):
    for c in range(SIZE):
        ax.add_patch(plt.Rectangle((c, SIZE-1-r), 1, 1, fill=False, edgecolor="gray"))
        if (r,c)==WALL: ax.add_patch(plt.Rectangle((c,SIZE-1-r),1,1,color="lightgray"))
        label = "S" if (r,c)==START else "G" if (r,c)==GOAL else "#" if (r,c)==WALL else ""
        ax.text(c+.5,SIZE-r-.5,label,ha="center",va="center",fontsize=14)
ax.plot([c+.5 for r,c in route],[SIZE-r-.5 for r,c in route],marker="o",linewidth=2)
ax.set(xlim=(0,SIZE),ylim=(0,SIZE),aspect="equal",title="Greedy policy route")
ax.axis("off")
plt.show()

## <span style="color:#15803D">42–45 min | Individual exit ticket (3 minutes)</span>
**Work individually; submit answers before the coach ends the session.**

1. **Math:** For a nonterminal step with `old_q=2`, `reward=-1`, `next_best=6`, `alpha=0.5`, `gamma=0.9`, compute the updated value.
2. **Debug:** Name the two conceptual bugs in the original code and state why the original cell was preserved.
3. **Interpret:** Explain exploration versus exploitation, and name **one** observation or limitation of the trained Q-table.

**Coach assessment:** Collect the answers as evidence of outcomes 2–4; use the opening discussion and pair-work observation for outcome 1. The 45-minute coaching session ends here; Q&A follows separately.